In [4]:
from flowrec.data.yambda import YambdaDataConfig, YambdaDataModule

In [ ]:
cfg = YambdaDataConfig(
    use_hf=True,
    dataset_type="sequential",
    dataset_size="50m",
    event_file="listens",
    min_played_ratio_pct=50,
    split_strategy="global_time",
    global_val_ratio=0.1,
    global_test_ratio=0.1,
    min_user_interactions=1,
    max_history_len=50,
    batch_size=256,
    keep_raw_interactions=True,
    global_eval_target_mode="last_per_user",
    train_all_prefixes=False
)

dm = YambdaDataModule(cfg)
dm.setup()

print(dm.summary())
batch = next(iter(dm.train_dataloader()))
print(batch["history_item_ids"].shape)

In [ ]:
from flowrec.item_representation.yambda import YambdaAudioEmbeddingConfig, YambdaAudioEmbeddingBuilder

store = YambdaAudioEmbeddingBuilder(
    YambdaAudioEmbeddingConfig(
        embeddings_path="../data/raw/yambda/embeddings.parquet",
        embedding_column="normalized_embed",
    )
).build()

store.save("../artifacts/item_embeddings/yambda_audio_embeddings.pt")

In [1]:
from flowrec.item_representation.content_embeddings import ItemEmbeddingStore
store = ItemEmbeddingStore.load("../artifacts/item_embeddings/yambda_audio_embeddings.pt")

In [ ]:
from flowrec.item_representation.semantic_id import (
    RQVAEConfig,
    RQVAETrainConfig,
    RQVAEItemTokenizerConfig,
    RQVAEItemTokenizer,
)

tokenizer = RQVAEItemTokenizer(
    RQVAEItemTokenizerConfig(
        rqvae=RQVAEConfig(
            input_dim=store.embeddings.shape[1],
            latent_dim=32,
            num_codebooks=3,
            codebook_size=256,
        ),
        train=RQVAETrainConfig(epochs=50, device="cpu"),
        normalize="l2",
        collision_policy="append",
    )
)

tokenizer.fit(store.embeddings, store.item_ids)
tokenizer.save("../artifacts/tokenizers/yambda_rqvae")

1
2
3


In [6]:
cfg = YambdaDataConfig(
    use_hf=True,
    dataset_type="sequential",
    dataset_size="50m",
    event_file="listens",
    min_played_ratio_pct=50,
    split_strategy="global_time",
    global_val_ratio=0.1,
    global_test_ratio=0.1,
    min_user_interactions=1,
    max_history_len=50,
    batch_size=256,
    keep_raw_interactions=True,
    global_eval_target_mode="last_per_user",
    train_all_prefixes=False,
    semantic_id_path="../artifacts/tokenizers/yambda_rqvae/semantic_ids.json",
    
)

dm = YambdaDataModule(cfg)
dm.setup()

batch = next(iter(dm.train_dataloader()))
print(batch["history_token_ids"].shape)
print(batch["target_token_ids"].shape)

/opt/anaconda3/envs/synth/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch.Size([256, 50, 4])
torch.Size([256, 4])
